# 09 · Deep unknown-fault test (Q3): LOCO-retrained CNN-6 and the three-way decision rule

## Purpose
Q3 asks whether the system notices a fault type it was never trained on. Each of the 9 fault classes is withheld in turn
(leave-one-class-out, LOCO); a CNN-6 (vibration only) is retrained without it, and the question is whether its recordings
are flagged **unknown** rather than passed off as a known fault or as healthy. The three-way rule turns scores into a
verdict: **healthy**, **a known fault**, or **unknown**. All numbers are **development findings** on the development pool,
not reserved-test results; one training seed in Stage 3C (seeds 43 and 44 added in Stage 3C.1, section 10).

> **On these development folds, embedding distance separates withheld bearing classes more successfully than withheld
> imbalance and vertical misalignment. When either of those two classes is withheld, the model frequently assigns the
> other label. The existing class labels are kept. This is a limitation of unfamiliar-fault detection, not a new physical
> diagnosis or merged class.** Withheld horizontal misalignment is sometimes declared healthy (see the healthy-rule
> variants, section 8). Development results on one partition; reserved tests sealed.

## Protocol
```
for each withheld class c (9) and outer dev fold k (3):                       27 runs
  known classes (≠ c), folds ≠ k ──(saved Stage 2D.1 Q3 split, 4:1:1 within each folder)──┐
      ├── fit          → train CNN-6 (9 outputs) and, on the fit NORMALS only, AE-6 (frame level)
      │                   norm_stats in development mode on fit (context excludes everything below and all of c)
      ├── calibration  → thresholds τ_p (MSP), τ_d (Mahalanobis): 97.5 % marginal targets; τ_e = max calibration-normal AE error
      └── seen-folder evaluation (a)
  fold k: known classes (b, unseen folders; normal is within-folder) and the withheld class c   → scored once per c
```
- **Scores** (directions fixed before running): MSP (lower = more unknown); Mahalanobis distance in the 128-d embedding
  (class centroids from fit windows, one shared Ledoit–Wolf covariance of within-class residuals, minimum over known
  classes per window, mean over 9 windows); k-NN (cosine distance to the 10th nearest fit window, mean over windows);
  AE reconstruction error. *AE error measures departure from healthy behaviour, not unfamiliarity among fault types; a
  familiar severe fault can have a larger error than an unfamiliar mild one.*
- **AE per (c, k), not per k:** the saved Q3 assignment splits the normal folder differently for every withheld class, so
  each run retrains its own AE-6 on its own fit normals (CLAUDE.md §7).
- **Three-way rule, PRIMARY = classifier-first** (declared before running; the AE misses some mild faults, and AE-first
  would declare them healthy): confident (MSP ≥ τ_p and d ≤ τ_d) → healthy (top class normal) or that known fault;
  otherwise unknown, except top class normal with AE error ≤ τ_e → healthy. **AE-first** is a comparison only.
- **Classical comparison:** logistic regression / MSP of Stage 2D.1 on the same split and recordings, re-run per
  recording and checked against its saved per-fold results. Observed differences only; no winner claims without paired
  intervals.

Stage 3A/3B (Q2-only) embeddings are not used here.

## Inputs
`results/loco/*.json|csv`, `results/scores/cnn6_w-<c>_s42_loco_fold<k>_recording.parquet`,
`results/scores/cnn6ae6_s42_loco_decisions.parquet`, `results/scores/logreg_s42_loco_decisions.parquet`, MLflow
runs (`stage = 3C`), all written by `scripts/09_loco_cnn.py`.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the settings, <code>metadata.csv</code> and the Stage 3C results, and stops if they were produced under different split assignments.</div>

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.cache import check_split_hash
from ml.config import load_config
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, RunPaths
from ml.splits import FAULT_CLASSES

cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
RES = SYSTEM_CODE_DIR / "results" / "loco"
FIG = SYSTEM_CODE_DIR / "results" / "figures"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
auc = json.loads((RES / "loco_auroc_oscr.json").read_text())
rule = json.loads((RES / "decision_rule.json").read_text())
for name, d in (("loco_auroc_oscr.json", auc), ("decision_rule.json", rule)):
    check_split_hash(d["provenance"]["split_hash"], meta, name)
dec = pd.read_parquet(RunPaths("loco", "cnn6ae6", 42, protocol="loco").scores("decisions"))
lr = pd.read_parquet(RunPaths("loco", "logreg", 42, protocol="loco").scores("decisions"))
per_fold = pd.read_csv(RES / "loco_per_fold_auroc_oscr.csv")
AN = RES / "analysis"
cnt = json.loads((AN / "item1_outcomes_counts.json").read_text())
assert cnt["recomputed_equals_saved_3C_decisions"]
print(auc["training"])
print("LR reproduction exact:", rule["lr_reproduction"]["exact"])

## 1. Score distributions: fold-k known vs the withheld class
For each withheld class (rows) and score (columns): the fold-k known recordings (grey) against the withheld recordings
(colour), pooled over the three folds. Pooling mixes three fold models, so this figure is descriptive; the AUROC table
below is computed within fold.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws, for every withheld class and every score, histograms of the known fold-k recordings (grey) and of the withheld class (colour). MSP is shown as 1 − MSP on a log axis so that, like the other scores, larger means more unknown.</div>

In [ ]:
scores = {"1 − MSP": lambda d: np.clip(1 - d.msp, 1e-7, None), "Mahalanobis": lambda d: d.maha,
          "k-NN (cosine, 10th)": lambda d: np.clip(d.knn, 1e-6, None), "AE error": lambda d: d.ae}
fig, axes = plt.subplots(len(FAULT_CLASSES), 4, figsize=(15, 2.0 * len(FAULT_CLASSES)))
for i, c in enumerate(FAULT_CLASSES):
    g = dec[dec.withheld == c]
    for j, (name, f) in enumerate(scores.items()):
        ax = axes[i, j]
        kn, un = f(g[g.role == "foldk_known"]), f(g[g.role == "foldk_withheld"])
        lo, hi = min(kn.min(), un.min()), max(kn.max(), un.max())
        bins = np.logspace(np.log10(lo), np.log10(hi), 30)
        ax.hist(kn, bins=bins, color="grey", alpha=0.6, density=True)
        ax.hist(un, bins=bins, color="C3", alpha=0.6, density=True)
        ax.set_xscale("log")
        ax.xaxis.set_minor_formatter(plt.NullFormatter())
        ax.tick_params(labelsize=6)
        if i == 0:
            ax.set_title(name, fontsize=9)
        if j == 0:
            ax.set_ylabel(c.replace("_misalignment", "_mis"), fontsize=8)
fig.suptitle("fold-k known (grey) vs withheld class (red), pooled over folds; density", y=1.0)
fig.tight_layout()
fig.savefig(FIG / "09_loco_score_distributions.png")

## 2. Threshold-free results: AUROC and OSCR
Per withheld class: (fold-k known) vs (fold-k withheld), mean ± std over the three folds; macro = mean over the 9
classes. OSCR uses the CNN's closed-set correctness on the known recordings.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Builds the AUROC and OSCR tables: one row per withheld class, one column per score, fold mean ± std; then the macro summary per score.</div>

In [ ]:
pc = pd.read_csv(RES / "loco_per_class_auroc_oscr.csv")
fmt = lambda m, s: f"{m:.3f} ± {s:.3f}"
display(pc.assign(v=[fmt(m, s) for m, s in zip(pc.auroc_mean, pc.auroc_std)]).pivot(index="withheld", columns="score", values="v")
        [["msp", "maha", "knn", "ae"]].rename_axis(None, axis=1).rename(columns=lambda x: f"AUROC {x}"))
display(pc.assign(v=[fmt(m, s) for m, s in zip(pc.oscr_mean, pc.oscr_std)]).pivot(index="withheld", columns="score", values="v")
        [["msp", "maha", "knn", "ae"]].rename_axis(None, axis=1).rename(columns=lambda x: f"OSCR {x}"))
pd.DataFrame(auc["summary"]).T.round(3)

## 3. Measured joint acceptance
τ_p and τ_d are each set for 97.5 % marginal acceptance on the calibration known recordings; the joint acceptance (both
accepted) is not assumed but measured on calibration, seen-folder evaluation and unseen-folder (fold-k) known recordings.
Mean and std over the 27 runs.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates the measured acceptance of MSP alone, Mahalanobis alone and both together, for calibration, seen-folder and unseen-folder known recordings, split into all known, known faults and normal.</div>

In [ ]:
ja = pd.DataFrame(rule["joint_acceptance"]["per_group_mean_over_runs"])
ja["group"] = pd.Categorical(ja.group, ["calibration", "seen_folder_eval", "unseen_foldk"])
ja.sort_values(["group", "subset"]).set_index(["group", "subset"]).round(3)

## 4. Outcomes, PRIMARY ordering (classifier-first)
Rates at the calibrated thresholds, in evaluation **instances** and **distinct recordings** (Stage 3C.1 correction): a
recording evaluated in several withheld-class runs counts once per run as an instance. For example, the fold-k normal row
is 225 evaluation instances from 25 distinct healthy recordings. Withheld recordings: healthy / known fault / unknown.
Known faults: seen-folder (a) and unseen-folder fold k (b), healthy / correct / wrong fault / unknown. Normal recordings:
healthy / fault / unknown. Two spreads are shown: the std over the 27 class × fold runs (mixes between-class and
between-fold variation; not seed uncertainty) and the std over the 3 folds of the per-fold macro value.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Defines a helper that turns the Stage 3C.1 outcome summaries into a table with instances, distinct recordings, rates and both spreads, then shows the primary (classifier-first) outcomes for every group, per withheld class, and the per-class false-unknown rates for (a) and (b).</div>

In [ ]:
STD27, STD3 = cnt["labels"]["std_27"], cnt["labels"]["std_3"]


def outcome_table(t):
    """One row per outcome group and outcome: instances / all instances, distinct recordings / all distinct, the pooled
    rate, the std over the 27 runs and the std over the 3 folds of the per-fold macro."""
    rows = []
    for g, e in t.items():
        for cat, p in e["pooled"].items():
            rows.append({"group": g, "outcome": cat, "instances": f"{p['instances']} / {e['instances']}",
                         "distinct recordings": f"{p['distinct_recordings']} / {e['distinct_recordings']}",
                         "rate (instances)": round(p["rate"], 3), "std, 27 runs": round(e[STD27][cat], 3),
                         "std, 3 folds (macro)": round(e[STD3][cat], 3)})
    return pd.DataFrame(rows).set_index(["group", "outcome"])


def per_class(t, group):
    return pd.DataFrame(t[group]["per_class"]).T.round(3)


print("std, 27 runs =", STD27)
print("std, 3 folds =", STD3)
A = cnt["cnn_primary_A"]
display(outcome_table(A))
display(per_class(A, "withheld"))
print("macro over withheld classes:", {k: round(v, 3) for k, v in A["withheld"]["macro_over_withheld_classes"].items()})
pd.DataFrame({g: {c: v["unknown"] for c, v in A[g]["per_class"].items()}
              for g in ("known_a_seen_folder", "known_b_unseen_foldk")}).round(3).rename(columns=lambda x: f"false unknown {x}")

### AE override (classifier-first)
How often the AE exception changes *unknown* to *healthy* (top class normal, not confident, AE error ≤ τ_e), per fold,
separately for truly healthy, known-fault and withheld-fault recordings. An override on a fault recording is a fault
declared healthy by the tie-break.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates the AE overrides per fold and recording kind (counts and percentages), for fold-k and seen-folder recordings.</div>

In [ ]:
ov = pd.DataFrame(rule["override_counts"]["per_fold"])
ov.pivot_table(index=["role", "kind"], columns="outer_fold", values=["overrides", "n"], aggfunc="sum")

## 5. Comparison ordering (AE-first)
Reported for comparison only; the primary ordering was declared before running.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the same outcome tables for the AE-first ordering.</div>

In [ ]:
display(outcome_table(cnt["cnn_ae_first"]))
per_class(cnt["cnn_ae_first"], "withheld")

## 6. Classical comparison (logistic regression / MSP, Stage 2D.1)
Same split, same recordings. Within-fold AUROC and OSCR differences for MSP (CNN − LR), and the practical outcomes at
each method's own thresholds (LR: MSP at its 95 % calibration target; CNN: the primary three-way rule; supplementary:
CNN MSP alone at 95 %). Observed differences; no winner claims without paired intervals.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Plots the per-run AUROC difference (CNN MSP − LR MSP) for every withheld class and fold, shows the LR/MSP outcome table in instances and distinct recordings, then tabulates the practical outcomes side by side: share of withheld faults passing as a known fault, declared healthy or unknown, and the false-unknown rates (a), (b) with the folder gap.</div>

In [ ]:
d = pd.read_csv(RES / "cnn_minus_lr_msp_per_fold.csv")
fig, ax = plt.subplots(figsize=(10, 3.6))
for k in range(3):
    g = d[d.outer_fold == k].set_index("withheld").loc[list(FAULT_CLASSES)]
    ax.scatter(np.arange(9) + (k - 1) * 0.15, g.auroc_diff, label=f"fold {k}", s=25)
ax.axhline(0, color="k", lw=0.7)
ax.set_xticks(range(9), [c.replace("_misalignment", "_mis") for c in FAULT_CLASSES], rotation=30, ha="right", fontsize=8)
ax.set(ylabel="AUROC difference (CNN − LR), MSP", title="Within-fold MSP AUROC difference per withheld class and fold")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "09_loco_cnn_minus_lr.png")
cc = rule["classical_comparison"]
print({k: cc["auroc_msp_cnn_minus_lr"][k] for k in ("mean", "std", "runs_positive", "runs")})
display(outcome_table(cnt["lr_msp"]))
pd.DataFrame({"CNN three-way (primary)": cc["practical_primary"], "CNN three-way (AE-first)": cc["practical_ae_first"],
              "LR / MSP (Stage 2D.1)": cc["practical_classical_lr"],
              "CNN MSP only, 95 % (suppl.)": cc["practical_cnn_msp_only_95"]}).round(3)

## 7. Stage 3C.1 · Withheld class → assigned class
For each withheld class, the final decision of the CNN primary rule (A) and of LR/MSP on the same split. For withheld
recordings instances and distinct recordings coincide (each is evaluated only in its own class's run). **These tables show
what the model confuses, not the physical cause.** The counts read from the saved decisions in the Stage 3C.1 brief are
reproduced below.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the withheld → assigned-class tables for the CNN rule and for LR/MSP (rows: withheld class; columns: final decision), then the reproduction check of the counts from the brief and the post-hoc subset.</div>

In [ ]:
conf = json.loads((AN / "item2_withheld_confusion.json").read_text())
print(conf["wording"])
display(pd.DataFrame(conf["cnn_rule_A"]["instances"]).T.fillna(0).astype(int))
display(pd.DataFrame(conf["lr_msp"]["instances"]).T.fillna(0).astype(int))
display(pd.DataFrame(conf["expected_checks"]).set_index("check"))
conf["post_hoc_subset"]

### Exploratory: embedding distances in the imbalance-withheld runs
**Exploratory and descriptive only; no causal claims.** (i) Mahalanobis distance of the withheld imbalance recordings to
each known-class centroid (the 3C centroids and shared covariance; per window, then the recording mean). (ii) Distance
of the fold-k known bearing recordings to the centroid of the withheld imbalance windows of the same run, by added
imbalance mass. The development pool holds no 35 g bearing recordings (35 g is the reserved severity test), so only 0, 6
and 20 g appear.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates, averaged over the three imbalance-withheld runs, the median distance of withheld imbalance recordings to each known class and how often each class is the nearest; then the median distance of known bearing recordings to the withheld imbalance centroid, by class and added mass.</div>

In [ ]:
tk = pd.read_csv(AN / "item2_imbalance_to_known_centroids.csv")
display(tk.groupby("known_class")[["median", "nearest_share"]].mean().sort_values("median").round(2))
bt = pd.read_csv(AN / "item2_bearing_to_withheld_imbalance.csv")
bt.pivot_table(index="class10", columns="added_mass_g", values="median").round(1)

## 8. Stage 3C.1 · Healthy-rule variants (saved scores, thresholds exactly as in 3C)
"Both classifier checks pass" = MSP ≥ τ_p AND Mahalanobis ≤ τ_d; "AE healthy" = AE error ≤ τ_e.
- **A** (3C primary): top prediction normal and both checks pass → healthy; top prediction a fault and both checks pass →
  that fault; otherwise unknown, except the AE override (top prediction normal AND AE healthy → healthy).
- **B**: A without the AE override.
- **C** (identified post hoc during review on 27 Sep; **exploratory**): healthy only if the top prediction is normal, both
  checks pass AND AE healthy; a fault with both checks passing → that fault; otherwise unknown.

For B and C the table below lists, relative to A, the withheld-fault → healthy instances removed and the normal → unknown
instances added. No ratio between them is computed or interpreted: **the normal instances repeat 25 distinct healthy
recordings**, and τ_e rests on only 2–3 calibration normals per run. The app's healthy rule is pending Bhuvesh's decision
after 3C.1, under stated assumptions about fault frequency and the relative cost of missed faults vs unnecessary
inspections; it is not inferred from these pooled counts.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows every outcome group for variants A, B and C side by side (instances, with distinct recordings in a second table), then the changes of B and C relative to A.</div>

In [ ]:
var = json.loads((AN / "item3_healthy_rule_variants.json").read_text())
vt = pd.read_csv(AN / "item3_healthy_rule_variants.csv")
fmt_ = lambda r: f"{r.instances} / {r.of}"
display(vt.assign(v=vt.apply(fmt_, axis=1)).pivot(index=["group", "outcome"], columns="variant", values="v"))
display(vt.assign(v=[f"{a} / {b}" for a, b in zip(vt.distinct, vt.of_distinct)])
        .pivot(index=["group", "outcome"], columns="variant", values="v").rename(columns=lambda x: f"{x} (distinct)"))
print(var["tau_e_limitation"], "|", var["normal_instances_note"])
pd.DataFrame(var["relative_to_A"])

## 9. Stage 3C.1 · CNN rule vs LR on the saved scores
**Threshold-free (primary):** per run, CNN Mahalanobis vs LR (−MSP) AUROC and OSCR (withheld vs fold-k known; OSCR
correctness = each method's own top class), paired differences, and a paired bootstrap: one stratified mixed-unit draw
over the whole development pool (fault folders within class, normal recordings individually) weights every run's
recordings by their multiplicity, so recordings reused across withheld-class runs are resampled jointly; macro over
classes of the fold mean per draw (runs left without withheld or known instances in a draw are dropped from that draw).
**An interval excluding zero supports a specific metric difference under this protocol, not an overall winner.**

**Operating point matched on calibration only:** per run, LR's threshold accepts as many calibration recordings as pass
both CNN checks; the rates are then measured on the evaluation groups.

**Retrospective diagnostic:** LR's false-unknown count matched to the CNN rule's directly on evaluation group (a), then
(b). *Retrospective and exploratory; these thresholds must not be deployed or used for model selection.*

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the threshold-free comparison (macro values, paired difference, bootstrap interval, runs where the CNN is higher), the calibration-matching summary, and the outcome counts of the CNN rule, LR at the calibration-matched threshold and the two retrospective LR thresholds.</div>

In [ ]:
cmpx = json.loads((AN / "item4_cnn_vs_lr.json").read_text())
print(cmpx["wording"])
tf = cmpx["threshold_free"]
display(pd.DataFrame({m: {k: (v if not isinstance(v, list) else f"[{v[0]:+.3f}, {v[1]:+.3f}]") for k, v in tf[m].items()}
                      for m in ("auroc", "oscr")}))
print(tf["bootstrap"])
op = cmpx["calibration_matched_operating_point"]
print(op["rule"]); print(op["summary"])
counts = lambda t: pd.DataFrame({g: {k: v["instances"] for k, v in e["pooled"].items()} for g, e in t.items()}).T
tabs = {"CNN rule A": op["cnn_rule_A"], "LR, calibration-matched": op["lr_matched"],
        "LR, retrospective (a) [do not deploy]": cmpx["retrospective_diagnostic"]["a_seen_folder"]["outcomes"],
        "LR, retrospective (b) [do not deploy]": cmpx["retrospective_diagnostic"]["b_unseen_foldk"]["outcomes"]}
pd.concat({k: counts(v) for k, v in tabs.items()}, axis=1).fillna("")

## 10. Stage 3C.1 · Seed variability (seeds 42, 43, 44)
The 27 LOCO CNN-6 and AE-6 runs were retrained with seeds 43 and 44 on the same saved split and settings; seed 42 was not
retrained (its metrics are recomputed from the saved outputs and checked against the 3C report). **Seeds measure training
variability only; they add no independent healthy or fault recordings.**

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the check of the recomputed seed-42 metrics against the 3C report, then the threshold-free macro AUROC/OSCR per score and the variant A/B/C outcome counts for each seed, with the mean and std over the three seeds, and the main withheld → assigned-class counts per seed.</div>

In [ ]:
sv = json.loads((AN / "item5_seed_variability.json").read_text())
print(sv["wording"]); print("seed 42 vs 3C report:", sv["seed42_check_against_3c"])
display(pd.DataFrame({k: v["values"] + [v["mean"], v["std"]] for k, v in sv["across_seeds"].items()},
                     index=["seed 42", "seed 43", "seed 44", "mean", "std"]).T.round(3))
pd.DataFrame({f"seed {s}": {"imbalance → vertical_misalignment": v["withheld_to_assigned_A"]["imbalance"].get("vertical_misalignment", 0),
                            "vertical_misalignment → imbalance": v["withheld_to_assigned_A"]["vertical_misalignment"].get("imbalance", 0),
                            "horizontal_misalignment → healthy": v["withheld_to_assigned_A"]["horizontal_misalignment"].get("healthy", 0)}
              for s, v in sv["per_seed"].items()})

## Findings
Development results on one partition; reserved tests sealed. On these development folds, embedding distance
separates withheld bearing classes more successfully than withheld imbalance and vertical misalignment. When either of
those two classes is withheld, the model frequently assigns the other label. The existing class labels are kept. This is
a limitation of unfamiliar-fault detection, not a new physical diagnosis or merged class. Withheld horizontal
misalignment is sometimes declared healthy (see the healthy-rule variants).
 27 LOCO runs (9 withheld classes × 3 folds). Counts are
evaluation **instances** and **distinct recordings**: a recording evaluated in several withheld-class runs counts once per
run as an instance.
- **Embedding distances separate unknown faults best; MSP less so; AE error does not.** Macro AUROC over the 9 withheld
  classes (fold-k known vs withheld): Mahalanobis 0.877 ± 0.194, k-NN 0.845 ± 0.174, MSP 0.782 ± 0.192, AE error
  0.517 ± 0.324 (± = std over the 9 class means; single runs 0.40–1.00, 0.44–1.00, 0.18–0.99, 0.06–1.00). The AE result
  is expected: its error measures departure from healthy behaviour, not unfamiliarity among fault types.
- **Joint acceptance is measured, not assumed.** With 97.5 % marginal calibration targets for τ_p and τ_d, the measured
  joint acceptance of known recordings is 0.988 on calibration, 0.972 on seen-folder evaluation and 0.864 on unseen-folder
  fold-k recordings.
- **Primary rule (classifier-first), withheld faults** (779 instances = 779 distinct recordings; each is evaluated only in
  its own class's run): 468 unknown (60.1 %; macro over withheld classes 70.2 %), **283 passed as a known fault
  (36.3 %)**, 28 declared healthy (3.6 %). Std over the 27 class × fold runs (mixes between-class and between-fold
  variation; not seed uncertainty) is 0.37 for unknown; the std over the 3 folds of the per-fold macro is 0.009.
- **Known faults:** false unknowns (a) seen folder 52 of 2,087 instances (35 of 738 distinct recordings; macro 2.3 %) vs
  (b) unseen folder 833 of 6,232 instances (253 of 779 distinct; macro 13.6 %); **1 known-fault instance (1 distinct
  recording) was declared healthy**, through the AE override (the 3C text said none: corrected); wrong-fault calls 2 and 7
  instances. **Normal:** fold k 192 of 225 instances healthy (225 evaluation instances from 25 distinct healthy
  recordings; 7 distinct recordings are unknown at least once), seen folder 68 of 74 instances (25 distinct); none called
  a fault.
- **AE override:** it turned unknown into healthy for 10 of 299 healthy instances (from 25 distinct recordings), 1 of
  8,319 known-fault instances and 9 of 779 withheld-fault instances (faults declared healthy by the tie-break). τ_e rests
  on only 2–3 calibration normals per run.
- **AE-first (comparison only):** fewer withheld faults called healthy (18 instances), but known faults are declared
  healthy (2.9–3.8 %) and 23–25 % of normal instances become unknown. The primary ordering was fixed before running.
- **Classical LR/MSP (same split and recordings), observed differences:** MSP AUROC CNN − LR −0.061 ± 0.182 over the 27
  runs (CNN higher in 12). At each method's own thresholds the CNN rule has fewer false unknowns ((a) macro 2.3 % vs
  4.2 %, (b) 13.6 % vs 23.7 %) but passes more withheld faults as known (283 vs 203 instances); healthy 28 vs 33. No
  winner is claimed without paired intervals.
- **Withheld → assigned class (what the model confuses, not the physical cause).** Withheld imbalance → vertical
  misalignment 139 of 144 instances (unknown 5); withheld vertical misalignment → imbalance 109 of 125 (unknown 12,
  healthy 4); withheld horizontal misalignment → healthy 22 (0.5 mm 10, 1.0 mm 7, 1.5 mm 5; 7 of the 22 through the AE
  override), unknown 51; overhang ball → underhang outer 11; underhang ball → overhang outer 11. LR/MSP on the same split:
  imbalance → vertical misalignment 97, vertical misalignment → imbalance 27, underhang cage → healthy 24.
- **Post-hoc subset (failure analysis, not a headline):** without imbalance and vertical misalignment as withheld
  classes, 451 of 510 withheld instances are unknown (88.4 %), 35 pass as known, 24 healthy.
- **Exploratory, descriptive only:** withheld imbalance recordings sit nearest the vertical-misalignment centroid in every
  run (median distance 18 vs ≥ 72 to every other class). Known bearing recordings with 20 g added mass do not sit
  consistently closer to the withheld imbalance centroid than 0 g recordings (closer for underhang outer race, farther
  for others); 35 g is not in the development pool.
- **Healthy-rule variants (saved scores, thresholds as in 3C).** Withheld faults declared healthy: A 28, B 19, C 8
  instances (C identified post hoc during review on 27 Sep; exploratory). Fold-k normals healthy: A 192, B 184, C 161 of
  225 instances; seen-folder normals: A 68, B 66, C 55 of 74. Relative to A, B removes 9 withheld → healthy instances and
  adds 8 + 2 normal → unknown instances; C removes 20 and adds 31 + 13 (14 and 9 distinct recordings). The normal instances
  repeat 25 distinct healthy recordings and τ_e rests on 2–3 calibration normals per run; no ratio is computed. The app's
  healthy rule is pending Bhuvesh's decision.
- **CNN rule vs LR, threshold-free (paired, saved scores):** CNN Mahalanobis − LR MSP macro AUROC +0.035 (paired
  mixed-unit bootstrap interval [+0.001, +0.071]; CNN higher in 22 of 27 runs) and OSCR +0.042 [+0.005, +0.081]. Both
  intervals exclude zero, which supports these specific metric differences under this protocol, not an overall winner.
  About 7 of 27 runs are dropped per draw on average (no withheld folder of that fold drawn).
- **At an LR threshold matched to the CNN rule's calibration acceptance** (exact in every run, no ties): LR sends more
  withheld faults to healthy (61 vs 28 instances) and to a known fault (331 vs 283), with similar false unknowns on known
  faults ((a) 45 vs 52, (b) 816 vs 833 instances) and more wrong-fault calls in (b) (78 vs 7). A retrospective diagnostic
  that matches LR's false unknowns directly on evaluation data is shown only for context and must not be deployed or used
  for model selection.
- **Seed variability (training variability only; no new recordings).** Seed 42 recomputed from saved outputs matches the
  3C report exactly. Macro AUROC over seeds 42 / 43 / 44: Mahalanobis 0.877 / 0.877 / 0.868 (mean 0.874 ± 0.005), k-NN
  0.846 ± 0.000, MSP 0.782 / 0.809 / 0.827 (0.806 ± 0.023), AE 0.515 ± 0.002. Rule A, withheld instances declared
  healthy 28 / 26 / 36, passed as known 283 / 269 / 261, unknown 468 / 484 / 482; fold-k normals healthy 192 / 189 / 191
  of 225. The imbalance ↔ vertical-misalignment confusion holds for every seed (139 / 138 / 134 and 109 / 100 / 101
  instances); withheld horizontal misalignment → healthy 22 / 23 / 33.


## Outputs written
- `results/figures/09_loco_score_distributions.png`, `09_loco_cnn_minus_lr.png`
- (by `scripts/09_loco_cnn.py`) `results/loco/*`; `results/scores/cnn6_w-<c>_s42_loco_fold<k>_*`,
  `cnn6ae6_s42_loco_decisions.parquet`, `logreg_s42_loco_decisions.parquet`; `artifacts/loco/<name>/fold<k>.pt`;
  MLflow runs tagged `stage = 3C`.